# NB_hospital_daily_summary (Silver → Gold)

**Target table:** `adb_caresync.gold.hospital_daily_summary`  
**Grain:** one row per hospital per calendar day  
**Load type:** full rebuild every run (`CREATE OR REPLACE TABLE ... AS SELECT`)

**How it works**
1. **daily** – group `silver.lab_results` by `hospital_id` and `test_date` to get the daily counts.
2. **daily_with_trend** – use `LAG()` to fetch the previous day's `total_tests` for the same hospital (ordered by date).
3. **Final select** – `LEFT JOIN` `silver.hospitals` (hospital name) and `silver.calendar` (day name, weekend flag), then calculate the rates and the day-over-day % change.

**Notes**
* The calendar join uses `date_key` (an INT like `20260115`), built from the test date with `date_format(..., 'yyyyMMdd')`.
* `LAG()` returns the previous row *with data* for that hospital. If a hospital had no tests on a day, that day has no row, so the comparison is with the last day that did have tests.
* `test_volume_change_pct` is `NULL` on a hospital's first day (no previous value) or when the previous value is 0.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab-test volumes, rates and day-over-day trend'
AS
-- Step 1: daily counts per hospital
WITH daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),

-- Step 2: bring in the previous day's total for the same hospital
daily_with_trend AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)

-- Step 3: add hospital + calendar attributes and the derived metrics
SELECT
  d.hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests            AS INT) AS total_tests,
  CAST(d.unique_patients_tested AS INT) AS unique_patients_tested,
  CAST(d.active_doctor_count    AS INT) AS active_doctor_count,
  CAST(d.abnormal_test_count    AS INT) AS abnormal_test_count,
  CAST(d.critical_test_count    AS INT) AS critical_test_count,
  CAST(d.abnormal_test_count / d.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(d.critical_test_count / d.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(d.prev_day_total_tests   AS INT) AS prev_day_total_tests,
  CASE
    WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
    ELSE CAST((d.total_tests - d.prev_day_total_tests) / d.prev_day_total_tests * 100 AS DECIMAL(6,2))
  END AS test_volume_change_pct
FROM daily_with_trend d
LEFT JOIN adb_caresync.silver.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT);

In [0]:
# %sql
# -- Each (hospital_id, summary_date) should appear once, and the total tests should match silver.lab_results
# SELECT
#   COUNT(*)                                              AS gold_rows,
#   COUNT(DISTINCT hospital_id, summary_date)             AS distinct_hospital_days,
#   SUM(total_tests)                                      AS total_tests,
#   (SELECT COUNT(*) FROM adb_caresync.silver.lab_results) AS silver_lab_rows,
#   COUNT_IF(day_name IS NULL)                            AS missing_calendar_rows
# FROM adb_caresync.gold.hospital_daily_summary;

In [0]:
# %sql
# SELECT * FROM adb_caresync.gold.hospital_daily_summary ORDER BY hospital_id, summary_date LIMIT 20;

In [0]:
# The table is built in a %sql cell, so there is no Python DataFrame.
# Read the gold table back into a DataFrame to get the row count.
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()


dbutils.notebook.exit(str(record_count))